# VML / AnyVisLoc — smoke (20 кадрів)

Етап 1 плану `docs/vml_implementation_plan.md`. Лише hosted Google Colab, без local runtime. Датасет завантажується в тимчасове сховище `/content/AnyVisLoc`; результати та checkpoints залишаються в приватному Drive. Це диск віртуальної машини Colab, не RAM і не диск вашого комп’ютера.

Synthetic Gaussian position prior погоджено для smoke. Attitude та AGL отримуються з dataset metadata/DSM; цей протокол не моделює їх похибки. 20 кадрів × 2 режими = 40 спроб. Навчання, калібрування R та validation gate ще не реалізовані.


In [3]:
import os
from pathlib import Path
if not os.environ.get('COLAB_RELEASE_TAG') or not Path('/content').is_dir():
    raise RuntimeError('Виберіть hosted Colab runtime. Локальне виконання заборонене.')
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Архів Scene 5 у Google Drive

Завантажте `Scene_05.zip` у власний Google Drive. Нижче вкажіть його шлях у `SCENE_ARCHIVE`; приклад — `Мій диск/AnyVisLoc/Scene_05.zip`. Шляхи до коду `REPO_ROOT` та результатів `OUTPUT_ROOT` залишаються вашими.

Архів читається безпосередньо зі змонтованого Drive і розпаковується в `/content/AnyVisLoc`. Друга копія ZIP у runtime не створюється; потрібне місце лише для розпакованої сцени. Архів у Drive автоматично не видаляється. OneDrive та cURL у цьому варіанті не використовуються.

Після втрати VM повторіть mount Drive і розпакування; завершені VML-спроби відновляться з checkpoints у Drive. `sha256` можна заповнити за наявності checksum від джерела. Фактичний SHA-256 архіву та hashes dataset-файлів зберігаються окремо.

Не змінюйте `pose_world_frame` навмання: runner перевіряє translation pose проти xyz.


In [13]:
REPO_ROOT = Path('/content/drive/MyDrive/PhD/proj')
# ZIP source in Drive:
SCENE_ARCHIVE = Path('/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_05.zip')
# Directory for extracted files on the Colab runtime disk:
DATASET_ROOT = Path('/content/AnyVisLoc')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PhD/proj/vml/anyvisloc-smoke-v3')
DATASET_ARCHIVES = [{'path': str(SCENE_ARCHIVE), 'sha256': ''}]

if not (REPO_ROOT / 'scripts').is_dir():
    raise FileNotFoundError(f'REPO_ROOT: немає папки scripts: {REPO_ROOT}')

drive_root = Path('/content/drive').resolve()
SCENE_ARCHIVE = SCENE_ARCHIVE.resolve()
if not SCENE_ARCHIVE.is_relative_to(drive_root):
    raise ValueError('SCENE_ARCHIVE — шлях до ZIP у Google Drive')
if not SCENE_ARCHIVE.is_file():
    raise FileNotFoundError(f'Архів не знайдено: {SCENE_ARCHIVE}')

DATASET_ROOT = DATASET_ROOT.resolve()
if DATASET_ROOT.suffix.lower() == '.zip':
    raise ValueError('DATASET_ROOT — папка розпакування, не ZIP. Шлях до ZIP вкажіть у SCENE_ARCHIVE')
if not DATASET_ROOT.is_relative_to(Path('/content')) or DATASET_ROOT.is_relative_to(drive_root):
    raise ValueError('DATASET_ROOT — папка runtime, наприклад /content/AnyVisLoc; ZIP залишається у Drive')
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
if not OUTPUT_ROOT.resolve().is_relative_to(drive_root):
    raise ValueError('OUTPUT_ROOT має бути у Google Drive')

print(f'Архів у Drive: {SCENE_ARCHIVE}')
print(f'Розпакування у runtime: {DATASET_ROOT}')


Архів у Drive: /content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_05.zip
Розпакування у runtime: /content/AnyVisLoc


## Перевірка та розпакування архіву

Перевіряється ZIP, SHA-256 і доступне місце на диску runtime. Архів читається зі Drive потоком, без завантаження цілком у RAM. Розпаковані файли залишаються в `/content/AnyVisLoc`; receipt і результати записуються в Drive. Архів-джерело зберігається.

Цю клітинку можна пропустити після reconnect, якщо runtime зберігся і сцена вже повністю розпакована.


In [5]:
import hashlib
import json
import shutil
import stat
import zipfile

receipts = []
for index, source in enumerate(DATASET_ARCHIVES):
    archive_path = Path(source['path']).expanduser().resolve()
    if not archive_path.is_file():
        raise FileNotFoundError(f'Архів не знайдено: {archive_path}')
    if not archive_path.is_relative_to(Path('/content/drive').resolve()):
        raise ValueError('Архів-джерело цього режиму має бути у змонтованому Drive')
    if not zipfile.is_zipfile(archive_path):
        raise ValueError('Файл не є ZIP. Перевірте завершення завантаження у Drive та формат архіву')
    hasher = hashlib.sha256()
    with archive_path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            hasher.update(block)
    actual_hash = hasher.hexdigest()
    expected_hash = source.get('sha256', '').strip().lower()
    if expected_hash and actual_hash != expected_hash:
        raise ValueError('SHA-256 архіву не збігається з очікуваним')
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        required_space = sum(member.file_size for member in members)
        if required_space >= shutil.disk_usage(DATASET_ROOT).free:
            raise RuntimeError(f'Для розпакування потрібно приблизно {required_space / 2**30:.1f} GiB; недостатньо місця у runtime')
        for member in members:
            target = (DATASET_ROOT / member.filename).resolve()
            if not target.is_relative_to(DATASET_ROOT.resolve()):
                raise ValueError('ZIP містить шлях поза DATASET_ROOT')
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('ZIP із symlink не підтримується')
        archive.extractall(DATASET_ROOT)
    receipts.append({'archive_index': index, 'archive_name': archive_path.name,
                     'sha256': actual_hash, 'archive_bytes': archive_path.stat().st_size,
                     'uncompressed_bytes': required_space})
    print(f'{archive_path.name}: розпаковано {required_space / 2**30:.2f} GiB; архів у Drive збережено')

references = sorted(DATASET_ROOT.rglob('L??_reference.json'))
assert references, 'У розпакованих даних немає Lxx_reference.json'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
receipt_path = OUTPUT_ROOT / 'dataset_staging_receipt.json'
receipt_tmp = receipt_path.with_suffix('.json.tmp')
receipt_tmp.write_text(json.dumps({'source_storage': 'google_drive',
                                  'dataset_storage': 'ephemeral_colab_disk',
                                  'archives': receipts}, indent=2), encoding='utf-8')
receipt_tmp.replace(receipt_path)
print(f'Знайдено reference JSON: {len(references)}; вільно {shutil.disk_usage(DATASET_ROOT).free / 2**30:.1f} GiB')


Scene_05.zip: розпаковано 2.19 GiB; архів у Drive збережено
Знайдено reference JSON: 1; вільно 82.3 GiB


In [14]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')])


0

In [15]:
import json
import sys
import importlib
from pathlib import Path

CONFIG_PATH = REPO_ROOT / 'configs/chance_constrained/vml/anyvisloc_smoke.v1.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8-sig'))
if config.get('pose_world_frame') != 'anyvisloc_ned_attitude':
    raise ValueError('Old Drive config: manually copy the updated anyvisloc_smoke.v1.json first')

sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()
prefix = 'scripts.pipeline.chance_constrained.visual_map_localization'
anyvisloc_adapter = importlib.import_module(prefix + '.anyvisloc')
importlib.reload(anyvisloc_adapter)
if not hasattr(anyvisloc_adapter, 'anyvisloc_attitude_to_enu'):
    raise RuntimeError('Old Drive adapter: manually copy the updated anyvisloc.py first')
planar_geometry = importlib.import_module(prefix + '.planar')
importlib.reload(planar_geometry)
smoke_localizer = importlib.import_module(prefix + '.smoke_localizer')
importlib.reload(smoke_localizer)
smoke_runner = importlib.import_module(prefix + '.anyvisloc_smoke')
importlib.reload(smoke_runner)
run_smoke = smoke_runner.run_smoke
print('Config:', CONFIG_PATH)
print('pose_world_frame:', config['pose_world_frame'])
print('Adapter:', anyvisloc_adapter.__file__)


Config: /content/drive/MyDrive/PhD/proj/configs/chance_constrained/vml/anyvisloc_smoke.v1.json
pose_world_frame: anyvisloc_ned_attitude
Adapter: /content/drive/MyDrive/PhD/proj/scripts/pipeline/chance_constrained/visual_map_localization/anyvisloc.py


## Діагностика конвенцій pose / xyz

Збережений вивід Scene 5 підтвердив, що translation дорівнює сирому `xyz`, а rotation відповідає ZYX-матриці з `euler_deg`. Конфігурація тепер явно використовує `anyvisloc_ned_attitude`: dataset-local translation і NED/FRD attitude перетворюються окремо; OpenCV-камера при нульовому attitude дивиться в надир, верх зображення — на північ. Це узгоджується з upstream pitch/yaw footprint; вирівнювання на реальних кадрах ще потрібно перевірити smoke-запуском.

Клітинка нижче читає лише числові метадані першого, середнього та останнього кадру. Збережений попередній вивід може ще містити старе `enu`; повторний запуск покаже поточну конфігурацію. Вивід можна прочитати зі збереженого notebook у workspace без копіювання у чат.


In [16]:
import numpy as np
import json

sample_paths = sorted(DATASET_ROOT.rglob('L??_????.npz'))
if not sample_paths:
    raise FileNotFoundError('No sample NPZ files under DATASET_ROOT')
pose_report = []
for sample_index in sorted({0, len(sample_paths) // 2, len(sample_paths) - 1}):
    sample_path = sample_paths[sample_index]
    with np.load(sample_path, allow_pickle=False) as sample_data:
        xyz_raw = np.asarray(sample_data['xyz'], dtype=float).reshape(3)
        c2w_raw = np.asarray(sample_data['pose_c2w'], dtype=float)
        item = {
            'sample': sample_path.relative_to(DATASET_ROOT).as_posix(),
            'xyz_raw': xyz_raw.tolist(),
            'xyz_enu_expected': (xyz_raw * [1, -1, 1]).tolist(),
            'pose_c2w_raw': c2w_raw.tolist(),
            'rotation_determinant': float(np.linalg.det(c2w_raw[:3, :3])),
            'translation_minus_xyz_raw': (c2w_raw[:3, 3] - xyz_raw).tolist(),
            'translation_minus_xyz_enu': (c2w_raw[:3, 3] - xyz_raw * [1, -1, 1]).tolist(),
        }
        if 'pose_w2c' in sample_data:
            w2c_raw = np.asarray(sample_data['pose_w2c'], dtype=float)
            item['inverse_max_error'] = float(np.max(np.abs(c2w_raw @ w2c_raw - np.eye(4))))
        if 'euler_deg' in sample_data:
            item['euler_deg_raw'] = np.asarray(sample_data['euler_deg'], dtype=float).tolist()
        pose_report.append(item)
print(json.dumps({'declared_pose_world_frame': config['pose_world_frame'],
                  'sample_count': len(sample_paths), 'samples': pose_report}, indent=2))


{
  "declared_pose_world_frame": "anyvisloc_ned_attitude",
  "sample_count": 1118,
  "samples": [
    {
      "sample": "Scene_05/L05_0001.npz",
      "xyz_raw": [
        244.86143493652344,
        492.378173828125,
        121.76885223388672
      ],
      "xyz_enu_expected": [
        244.86143493652344,
        -492.378173828125,
        121.76885223388672
      ],
      "pose_c2w_raw": [
        [
          -0.8945038318634033,
          0.44456857442855835,
          -0.04713517054915428,
          244.86143493652344
        ],
        [
          -0.4400913119316101,
          -0.8941956162452698,
          -0.08206028491258621,
          492.378173828125
        ],
        [
          -0.07862948626279831,
          -0.052659451961517334,
          0.9955121278762817,
          121.76885223388672
        ],
        [
          0.0,
          0.0,
          0.0,
          1.0
        ]
      ],
      "rotation_determinant": 1.0000000554699648,
      "translation_minus_xyz_raw":

## Запуск / відновлення

Після reconnect повторіть підключення Drive та підготовку коду. Якщо VM втрачено і `/content/AnyVisLoc` порожній, повторно розпакуйте той самий архів із Drive. Перевірені per-trial файли з Drive повторно не виконуються; shards і статуси відновлюються з них. Зміна коду, конфігурації чи версій бібліотек вимагає нової папки результатів. Пошкоджені файли викликають помилку checksum.

`altitude_eligible` означає тільки AGL 20–120 м. Геометричні відмови зберігаються як reject; smoke не замінює повний eligibility manifest для pilot. Наявність 20 придатних кадрів у Scene 5 ще потрібно підтвердити.


In [17]:
summary = run_smoke(DATASET_ROOT, OUTPUT_ROOT, config)
print(json.dumps(summary, indent=2))


{
  "aerial": {
    "trial_count": 20,
    "accepted_count": 4,
    "reject_count": 16,
    "acc_at_m": {
      "1": 0.2,
      "3": 0.2,
      "5": 0.2,
      "10": 0.2,
      "20": 0.2
    },
    "validation_passed": false
  },
  "satellite": {
    "trial_count": 20,
    "accepted_count": 0,
    "reject_count": 20,
    "acc_at_m": {
      "1": 0.0,
      "3": 0.0,
      "5": 0.0,
      "10": 0.0,
      "20": 0.0
    },
    "validation_passed": false
  }
}


In [18]:
manifest = json.loads((OUTPUT_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status'] == 'completed'
assert all(summary[m]['trial_count'] == 20 for m in ('aerial', 'satellite'))


{
  "R_calibrated": false,
  "campaign": "smoke",
  "config_fingerprint": "1e842a778f917fc8369a05af7d40d5e8f7c24b0d554c5410b4aafcd5f905f2b2",
  "dataset_files": {
    "Scene_05/L05_0001.npz": "cd3f260f9bd60b5eb1a8ccab9d7f21d92c5c6475b04b474f0c6c30d70f90ae17",
    "Scene_05/L05_0002.npz": "8625db1baa128124514327b0be4986580f705a551ae66a3f0103591b89a5babf",
    "Scene_05/L05_0003.npz": "21ad749e3fdaa2129e43f48a139f6e37d4eb430689c59510411dd04bc51a99ea",
    "Scene_05/L05_0004.npz": "ec0a935af59e8bd3f166cbf384fbf2d341dcd913d0d36023e55f6c61c16f2ee1",
    "Scene_05/L05_0005.npz": "cc4e27fdc269738419341f47298c28eca7032b7bffc4b6e3311c9a4dd8ed4679",
    "Scene_05/L05_0006.npz": "1768ab5e174070278b8d96e82317d9d335275303cf2b19c493c8a779875ae25d",
    "Scene_05/L05_0007.npz": "dcc0f04b5f902820e62e125560b6b94c3c00849808e35c85a6b5e3925e0a905b",
    "Scene_05/L05_0008.npz": "d381404b56800268c2ec51c3a1950caaa35751e28db4051613620e8d0010f9c8",
    "Scene_05/L05_0009.npz": "a71d92b68188e862fa176ae030b7c51

In [19]:
# Read retained results only: does not rerun matching or change thresholds.
from collections import Counter
import hashlib
import json

run_metadata = json.loads((OUTPUT_ROOT / 'run.json').read_text(encoding='utf-8'))
shard_paths = sorted((OUTPUT_ROOT / 'shards').rglob('trials.json'))
if not shard_paths:
    raise FileNotFoundError(f'No completed trial shards under {OUTPUT_ROOT}')
reason_counts = {'aerial': Counter(), 'satellite': Counter()}
examples = {}
for shard_path in shard_paths:
    status_path = shard_path.with_name('status.json')
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status['run_identity'] != run_metadata['identity']:
        raise ValueError('Shard belongs to a different run')
    shard_bytes = shard_path.read_bytes()
    if status.get('status') != 'completed' or hashlib.sha256(shard_bytes).hexdigest() != status.get('sha256'):
        raise ValueError(f'Incomplete or corrupted shard: {shard_path}')
    for trial in json.loads(shard_bytes):
        mode = trial['mode']
        result = trial['result']
        reason = result['reason']
        reason_counts[mode][reason] += 1
        example_key = mode + ':' + reason
        if example_key not in examples:
            examples[example_key] = {
                'sample': trial['sample'], 'agl_m': trial['agl_m'],
                'runtime_s': result['runtime_s'],
                'diagnostics': result['diagnostics'],
            }
report = {'output_dir': str(OUTPUT_ROOT),
          'reasons_by_mode': {mode: dict(counts) for mode, counts in reason_counts.items()},
          'one_example_per_reason': examples}
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "output_dir": "/content/drive/MyDrive/PhD/proj/vml/anyvisloc-smoke-v3",
  "reasons_by_mode": {
    "aerial": {
      "accepted_xy_uncalibrated": 4,
      "insufficient_map_coverage": 16
    },
    "satellite": {
      "insufficient_inliers": 3,
      "insufficient_map_coverage": 16,
      "insufficient_matches": 1
    }
  },
  "one_example_per_reason": {
    "aerial:accepted_xy_uncalibrated": {
      "sample": "Scene_05/L05_0001.npz",
      "agl_m": 104.45551872253418,
      "runtime_s": 0.9196631419999903,
      "diagnostics": {
        "R_calibrated": false,
        "R_method": "uncalibrated_translation_residual_approximation",
        "footprint_area_m2": 24652.93212784373,
        "good_matches": 89,
        "inlier_ratio": 0.9662921348314607,
        "inliers": 86,
        "map_coverage": 0.9539395734597157,
        "median_reprojection_error_px": 0.3813219666481018,
        "projection_condition": 1.0047854153608378,
        "query_keypoints": 1151,
        "query_shape_px": 

## Після smoke

Передайте `summary.json`, `manifest.json` та причини reject для аналізу. Наступний етап: повний inventory/геометрична eligibility, maximin/medoid pilot на 3 сценах, training-only tuning, residual mixture та post-match R, 5-fold GroupKFold окремо за режимами. Валідаційних імовірностей або planner-таблиць smoke не створює. `deployment_status=production_enabled` застосовується до найкращого **навченого** артефакту на наступному етапі; smoke не має навченого артефакту.